In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import pyarrow.parquet as pq

from tqdm import tqdm

import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

# Upload data

We are now working solely with 2010 data

In [2]:
path = "data/"

In [3]:
df_country = pd.read_csv(path+'countryInfo.csv')
id2name=dict(zip(df_country['alpha-2'],df_country['name']))
id2region=dict(zip(df_country['alpha-2'],df_country['region']))
id2subregion=dict(zip(df_country['alpha-2'],df_country['sub-region']))

id2name['XK']='Kosovo'
id2region['XK']='Europe'
id2subregion['XK']='Southern Europe'

In [4]:
df_topics=pd.read_csv(path+'topic_mapping_table_19022024.csv')
df_topics['topic_id']=df_topics['topic_id'].apply(lambda x: 'T'+str(x))
id2name=dict(zip(df_topics['topic_id'],df_topics['topic_name']))
id2field=dict(zip(df_topics['topic_id'],df_topics['field_name']))

In [5]:
parqToRead=path+'authorsPapersTopicsYearInstitutions.parquet'
# Read the parquet file filtering the lines where year=2005
table = pq.read_table(parqToRead, filters=[('year','=','2010')])
#table = pq.read_table(parqToRead)
# Convert to pandas DataFrame
df = table.to_pandas()

In [6]:
df.head()

,authors,id,year,institution,country,coord,inst_name,topic
83,A5104829885,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
84,A5010153016,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
85,A5061654139,W4400863937,2010,I189238007,UA,"[50.45466, 30.5238]",Taras Shevchenko National University of Kyiv,T12618
820,A5052243613,W2112665120,2010,I4210134612,GB,"[51.1891, -0.890375]",Forest Research,T12618
821,A5072218476,W2112665120,2010,I4210134612,GB,"[51.1891, -0.890375]",Forest Research,T12618


# Utility functions

In [42]:
def get_subfield_info(subfield_id: int, df_topics: pd.DataFrame = df_topics):
    return (
        df_topics
        [["subfield_id", "subfield_name", "field_name", "domain_name"]]
        .query(f"subfield_id == {subfield_id}")
        .head(1)
    )

def get_country_info(code: str, df_country: pd.DataFrame = df_country):
    return (
        df_country
        [["name", "region", "sub-region", "alpha-2"]]
        .rename(columns={"alpha-2": "code"})
        .query(f"code == \"{code}\"")
    )

# Aggregate by subfield

In [7]:
df_author_subfield = (
    df
    .merge(df_topics[["topic_id", "subfield_id"]], left_on="topic", right_on="topic_id", how="left")
    .drop(["topic_id"], axis=1)
)

In [8]:
df_country_subfield = (
    df_author_subfield
    [["id", "country", "subfield_id"]]
    .drop_duplicates()
    .groupby(["country", "subfield_id"], as_index=False)
    .count()
    .pivot(index="country", columns="subfield_id", values="id")
)
mx_country_subfield = np.matrix(df_country_subfield.values)
df_country_subfield

subfield_id,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE,1.0,3.0,1.0,3.0,1.0,6.0,NaN,NaN,NaN,13.0,...,NaN,4.0,2.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0
AF,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,...,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,3.0,2.0,1.0,NaN,NaN,2.0,...,NaN,2.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,2.0,NaN,1.0,NaN,1.0,NaN,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN
YE,NaN,3.0,2.0,1.0,1.0,3.0,NaN,NaN,1.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
ZA,30.0,51.0,52.0,16.0,214.0,77.0,17.0,NaN,66.0,290.0,...,6.0,27.0,8.0,2.0,NaN,16.0,16.0,1.0,14.0,13.0


In [9]:
df_country_subfield.to_csv(path+"df_country_subfield.csv")

In [280]:
# top-20 active countries
df_articles_by_country = (
    df_country_subfield
    .sum(axis=1)
    .to_frame(name='total_articles')
    .assign(log_total_articles=lambda df: np.log(df.total_articles))
)
(
    df_articles_by_country
    .sort_values("total_articles", ascending=False)
    # .head(10)
)

top_20_countries_list = df_articles_by_country.sort_values("total_articles", ascending=False).head(20).index.tolist()

In [285]:
px.histogram(df_articles_by_country, x="log_total_articles", title="Number of Articles log-distribution")

In [289]:
from scipy.stats import kstest, norm

# Example data
data = df_articles_by_country.log_total_articles.values

# KS test against standard normal
stat, p_value = kstest((data - data.mean()) / data.std(), 'norm')

print("KS statistic:", stat)
print("p-value:", p_value)

KS statistic: 0.054378986693755293
p-value: 0.5158849986810807


In [170]:
df_plot = (
    df_articles_by_country
    .merge(df_country[["alpha-2", "region"]], left_index=True, right_on="alpha-2", how="left")
    .sort_values("total_articles")
    .assign(total_cumsum=lambda df: df.total_articles.cumsum())
)
px.scatter(df_plot, x="alpha-2", y="total_cumsum", color="region",category_orders={"alpha-2": df_plot["alpha-2"].tolist()})

In [114]:
# top-10 active subfields (in the world)
df_subfield_representation_world = (
    df_country_subfield
    .sum(axis=0)
    .to_frame(name='total_articles')
    .assign(total_share=lambda df: df.total_articles / df.total_articles.sum())
)
(
    df_subfield_representation_world
    .sort_values("total_articles", ascending=False)
    .merge(df_topics[["subfield_id", "subfield_name"]].drop_duplicates(), left_index=True, right_on="subfield_id", how="left")
    .head(10)
)


,total_articles,total_share,subfield_id,subfield_name
17,119098.0,0.042790,2208,Electrical and Electronic Engineering
14,111396.0,0.040023,1312,Molecular Biology
54,80676.0,0.028985,3312,Sociology and Political Science
73,67400.0,0.024216,2505,Materials Chemistry
19,62314.0,0.022388,1702,Artificial Intelligence
39,55636.0,0.019989,2207,Control and Systems Engineering
188,54466.0,0.019569,2746,Surgery
58,53857.0,0.019350,2204,Biomedical Engineering
79,51858.0,0.018632,1705,Computer Networks and Communications
187,47028.0,0.016896,2210,Mechanical Engineering


In [123]:
df_country_subfield_norm = (
    df_country_subfield
    .div(df_country_subfield.sum(axis=1), axis=0)
)
df_country_subfield_norm

subfield_id,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE,0.000660,0.001979,0.000660,0.001979,0.000660,0.003958,NaN,NaN,NaN,0.008575,...,NaN,0.002639,0.001319,NaN,NaN,NaN,0.001319,NaN,NaN,0.001319
AF,NaN,0.027778,NaN,NaN,0.027778,NaN,NaN,NaN,NaN,0.027778,...,NaN,0.027778,0.027778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.062500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,0.013393,0.008929,0.004464,NaN,NaN,0.008929,...,NaN,0.008929,NaN,NaN,NaN,NaN,NaN,0.004464,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,0.019048,NaN,0.009524,NaN,0.009524,NaN,NaN,NaN,0.028571,...,NaN,NaN,NaN,NaN,NaN,NaN,0.009524,NaN,0.009524,NaN
YE,NaN,0.009934,0.006623,0.003311,0.003311,0.009934,NaN,NaN,0.003311,0.033113,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003311,NaN
ZA,0.002608,0.004434,0.004521,0.001391,0.018604,0.006694,0.001478,NaN,0.005738,0.025211,...,0.000522,0.002347,0.000695,0.000174,NaN,0.001391,0.001391,0.000087,0.001217,0.001130


In [265]:
df_country_subfield_norm_world = (
    df_country_subfield_norm
    .div(df_subfield_representation_world.total_share)
)
df_country_subfield_norm_world_norm = (
    df_country_subfield_norm_world
    .div(df_country_subfield_norm_world.sum(axis=1), axis=0)
)
df_country_subfield_norm_world_norm

subfield_id,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.199013,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE,0.002041,0.004267,0.001424,0.007188,0.000430,0.003052,NaN,NaN,NaN,0.002359,...,NaN,0.006413,0.006399,NaN,NaN,NaN,0.007203,NaN,NaN,0.005252
AF,NaN,0.051835,NaN,NaN,0.015685,NaN,NaN,NaN,NaN,0.006611,...,NaN,0.058422,0.116595,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.151868,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,0.009078,0.007153,0.039160,NaN,NaN,0.002551,...,NaN,0.022543,NaN,NaN,NaN,NaN,NaN,0.022650,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,0.032253,NaN,0.027165,NaN,0.005767,NaN,NaN,NaN,0.006171,...,NaN,NaN,NaN,NaN,NaN,NaN,0.040832,NaN,0.038867,NaN
YE,NaN,0.017513,0.011684,0.009834,0.001766,0.006263,NaN,NaN,0.004791,0.007446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014070,NaN
ZA,0.006312,0.007480,0.007633,0.003953,0.009498,0.004039,0.009764,NaN,0.007945,0.005425,...,0.00812,0.004464,0.002639,0.001861,NaN,0.006853,0.005942,0.000332,0.004949,0.003520


In [183]:
df_subfield_country_norm = (
    df_country_subfield
    .T
    .div(df_country_subfield.T.sum(axis=1), axis=0)
)
df_subfield_country_norm

country,AD,AE,AF,AG,AL,AM,AO,AR,AS,AT,...,VG,VI,VN,VU,WS,XK,YE,ZA,ZM,ZW
subfield_id,,,,,,,,,,,,,,,,,,,,,
1100,NaN,0.000273,NaN,NaN,NaN,NaN,0.000818,0.018003,NaN,0.004637,...,NaN,0.000273,0.000273,NaN,NaN,NaN,NaN,0.008183,NaN,0.000273
1102,NaN,0.000570,0.000190,NaN,NaN,0.000190,NaN,0.009888,NaN,0.005134,...,NaN,NaN,0.001521,NaN,NaN,0.000380,0.000570,0.009698,NaN,0.000761
1103,NaN,0.000190,NaN,NaN,NaN,NaN,NaN,0.005138,NaN,0.003996,...,NaN,NaN,0.001332,NaN,NaN,NaN,0.000381,0.009895,NaN,NaN
1104,NaN,0.000961,NaN,NaN,NaN,NaN,NaN,0.003844,NaN,0.001602,...,NaN,NaN,0.002562,NaN,NaN,0.000320,0.000320,0.005125,NaN,NaN
1105,NaN,0.000058,0.000058,NaN,0.000173,0.000288,NaN,0.016974,NaN,0.008631,...,NaN,NaN,0.000518,NaN,0.000058,NaN,0.000058,0.012313,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002221,NaN,0.001666,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008884,NaN,NaN
3611,NaN,0.000963,NaN,NaN,NaN,NaN,NaN,0.000963,NaN,0.004815,...,NaN,NaN,0.000481,NaN,NaN,0.000481,NaN,0.007703,NaN,NaN
3612,NaN,NaN,NaN,NaN,0.000431,NaN,NaN,0.001292,NaN,0.011197,...,NaN,NaN,0.000431,NaN,NaN,NaN,NaN,0.000431,NaN,NaN


In [213]:
fig = px.imshow(
    df_subfield_country_norm[top_20_countries_list].T.values,
    labels=dict(x="Country", y="Subfield", color="Representation vs The world"),
    y=top_20_countries_list,
    x=df_subfield_country_norm.index,
    color_continuous_scale="Viridis",  # or "Cividis", "Magma", etc.
    # range_color=[0, 1]
)
fig.update_layout(title="Weight of Country production in the sub-field in the world", xaxis_title="Country", yaxis_title="Subfield")
fig.show()

In [269]:
# # import plotly.express as px
#
# # Convert pivoted matrix to long format for plotly
# # matrix_long = matrix.reset_index().melt(id_vars=matrix.index.name, var_name="X", value_name="Value")
#
fig = px.imshow(
    df_country_subfield_norm_world.loc[top_20_countries_list].values,
    labels=dict(x="Country", y="Subfield", color="Representation vs The world"),
    x=df_country_subfield_norm_world.columns,
    y=top_20_countries_list,
    color_continuous_scale="Viridis",  # or "Cividis", "Magma", etc.
    range_color=[0, 2]
)
fig.update_layout(title="Relative Representation: 1 means exactly relative to the world", xaxis_title="X-axis", yaxis_title="Y-axis")
fig.show()

In [271]:
# # import plotly.express as px
#
# # Convert pivoted matrix to long format for plotly
# # matrix_long = matrix.reset_index().melt(id_vars=matrix.index.name, var_name="X", value_name="Value")
#
fig = px.imshow(
    df_country_subfield_norm_world.values,
    labels=dict(x="Country", y="Subfield", color="Representation vs The world"),
    x=df_country_subfield_norm_world.columns,
    y=df_country_subfield_norm_world.index,
    color_continuous_scale="Viridis",  # or "Cividis", "Magma", etc.
    range_color=[0, 2]
)
fig.update_layout(title="Relative Representation: 1 means exactly relative to the world", xaxis_title="X-axis", yaxis_title="Y-axis", height=1600,)
fig.show()

In [272]:
get_subfield_info(1602)

,subfield_id,subfield_name,field_name,domain_name
179,1602,Analytical Chemistry,Chemistry,Physical Sciences


In [246]:
get_country_info("AX")

,name,region,sub-region,code
1,Åland Islands,Europe,Northern Europe,AX


In [238]:
df_country_subfield_norm_world.max().max()

np.float64(9216.317880794702)

In [242]:
np.where(df_country_subfield_norm_world.values == df_country_subfield_norm_world.max().max())

(array([12]), array([193]))

In [243]:
df_country_subfield_norm_world.iloc[12, 193]

np.float64(9216.317880794702)

In [244]:
df_country_subfield_norm_world.index[12]

'AX'

In [245]:
df_country_subfield_norm_world.columns[193]

np.int64(2922)

In [251]:
df_country_subfield.loc["AX"].to_frame().dropna()

,AX
subfield_id,
2922,1.0
3317,1.0


In [255]:
df_subfield_representation_world.loc[2922]

total_articles    151.000000
total_share         0.000054
Name: 2922, dtype: float64